# Coding Exercise: Melbourne Housing Price Prediction

**Dataset:** `./datasets/housing/melb_data.csv` (Melbourne housing)

## Your Task
Build an end-to-end regression workflow to predict house prices using the same ideas from the demo notebook.

1. Load and inspect the data
2. Separate numerical and categorical columns
3. Define features + label (target)
4. Split into train/test
5. Build a preprocessing + model **Pipeline**
6. Evaluate with RMSE
7. Use cross-validation without data leakage
8. Tune with GridSearchCV

## Success Criteria
By the end, you should be able to:
- implement a leakage-safe `Pipeline(preprocess + model)`
- compare model quality using cross-validation RMSE
- report best hyperparameters and final test RMSE

### Rules
- Fill in the code where you see `# TODO:`.
- Do **not** fit preprocessing on the test set.
- For cross-validation and grid search, use a **single Pipeline(preprocess + model)** to avoid leakage.
- Keep `random_state=42` where requested so results are reproducible.

## 1) Imports
Run this cell to import the libraries you will need.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error

## 2) Load the data
Load `melb_data.csv` into a DataFrame named `df`. Then show the first 5 rows and the column names.

In [6]:
# TODO: load the Melbourne housing dataset
# Path: ./datasets/housing/melb_data.csv
import os

def load_housing_data(housing_path="./datasets/housing/"):
    csv_path = os.path.join(housing_path, "melb_data.csv")
    return pd.read_csv(csv_path)
df = load_housing_data()

# TODO: display basic info
df.head()

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


### Quick inspection
1. How many rows and columns are there?
2. Which columns have missing values?
3. Which column looks like the target (price)?

In [3]:
# TODO: inspect the dataset
df.shape

(13580, 21)

In [8]:
# TODO: show missing values per column (sorted)

missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percentage": missing_pct,
})

# Keep only columns that actually have missing values, sorted by highest percentage
missing_summary = missing_summary[missing_summary["missing_count"] > 0].sort_values(
    by="missing_percentage", ascending=False
)

missing_summary

,missing_count,missing_percentage
BuildingArea,6450,47.50
YearBuilt,5375,39.58
CouncilArea,1369,10.08
Car,62,0.46


## 3) Separate numerical and categorical columns
Before defining features and labels, identify which columns in the dataset are numerical and which are categorical. Use `select_dtypes()` on `df` (excluding the target column `Price`) to separate the columns:
- categorical columns with dtype `object`
- numerical columns with numeric dtypes

This follows the same pattern used in the demo notebook.

In [14]:
# TODO: identify categorical and numerical columns
categorical_cols = df.drop(columns=['Price']).select_dtypes(include=["object", "str"]).columns
numerical_cols = df.drop(columns=['Price']).select_dtypes(include=["number"]).columns

# Check
assert "Price" in df.columns, "Column 'Price' is missing from df."
categorical_cols = list(categorical_cols)
numerical_cols = list(numerical_cols)
all_cols = set(df.drop("Price", axis=1).columns)
assert set(categorical_cols).isdisjoint(set(numerical_cols)), "Categorical and numerical columns must not overlap."
assert set(categorical_cols).union(set(numerical_cols)) == all_cols, "Categorical + numerical columns must cover all feature columns (df without Price)."
print("Step 3 check passed.")

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

Step 3 check passed.
Categorical columns: ['Suburb', 'Address', 'Type', 'Method', 'SellerG', 'Date', 'CouncilArea', 'Regionname']
Numerical columns: ['Rooms', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'YearBuilt', 'Lattitude', 'Longtitude', 'Propertycount']


## 4) Define label (y) and features (X)
We will predict the column `Price`. Create:
- `y` = `df['Price']`
- `X` = all other columns (drop `Price`)

In [17]:
# TODO: define X and y
y = df['Price']
X = df.drop(columns=['Price'])


# Check
assert "Price" in df.columns, "Column 'Price' is missing from df."
assert isinstance(X, pd.DataFrame), "X should be a pandas DataFrame."
assert isinstance(y, pd.Series), "y should be a pandas Series."
assert "Price" not in X.columns, "X should not include the target column 'Price'."
assert len(X) == len(y) == len(df), "X, y, and df must have the same number of rows."
print("Step 4 check passed.")

X.head()

Step 4 check passed.


,Suburb,Address,Rooms,Type,Method,SellerG,Date,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,S,Biggin,3/12/2016,2.5,3067.0,2.0,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,S,Biggin,4/02/2016,2.5,3067.0,2.0,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,SP,Biggin,4/03/2017,2.5,3067.0,3.0,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,PI,Biggin,4/03/2017,2.5,3067.0,3.0,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,VB,Nelson,4/06/2016,2.5,3067.0,3.0,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


## 5) Train/test split
Split into train and test sets:
- 80% train, 20% test
- set `random_state=42`

Create variables: `X_train`, `X_test`, `y_train`, `y_test`.

In [19]:
# TODO: train/test split
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
y_train, y_test = train_test_split(y, test_size=0.2, random_state=42)

# Check
assert len(X_train) + len(X_test) == len(X), "Train and test rows should add up to total rows."
assert len(y_train) == len(X_train), "X_train and y_train must have the same number of rows."
assert len(y_test) == len(X_test), "X_test and y_test must have the same number of rows."
assert set(X_train.columns) == set(X.columns), "X_train columns should match original X columns."
assert len(set(X_train.index).intersection(set(X_test.index))) == 0, "Train and test sets should not overlap."
print("Step 5 check passed.")

print(X_train.shape, X_test.shape)

Step 5 check passed.
(10864, 20) (2716, 20)


## 6) Build preprocessing (ColumnTransformer)
Use the `categorical_cols` and `numerical_cols` you created earlier.

We will:
- **Numerical columns**: impute missing values (median) + scale (StandardScaler)
- **Categorical columns**: impute missing values (most_frequent) + one-hot encode

### Task
1. Create the numeric preprocessing pipeline.
2. Create the categorical preprocessing pipeline.
3. Build `preprocess = ColumnTransformer(...)` using the earlier column lists.

In [35]:
# TODO: create numeric and categorical preprocessing pipelines
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

#housing_num = y.select_dtypes(include=["number"]).copy()

numeric_transformer = Pipeline([
('imputer', SimpleImputer(strategy="median")),
('std_scaler', StandardScaler()),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# TODO: build the full ColumnTransformer
preprocess = ColumnTransformer([
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols),
])

# Check
assert isinstance(preprocess, ColumnTransformer), "preprocess must be a ColumnTransformer."
_ = preprocess.fit_transform(X_train.head(5), y_train.head(5))
print("Step 6b check passed.")

preprocess

Step 6b check passed.


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

## 7) Baseline model (Linear Regression) in a Pipeline
Create a pipeline called `lin_model` that includes:
- `preprocess`
- `LinearRegression()`

Then:
1. Fit on the training set
2. Predict on the test set
3. Compute RMSE on the test set

In [37]:
# TODO: create + fit the baseline pipeline
lin_model = Pipeline(steps=[
    ('preprocessor', preprocess),
    ('regressor', LinearRegression())
])

# TODO: fit on the training set
lin_model.fit(X_train, y_train)

# TODO: predict on the test set
lin_pred = lin_model.predict(X_test)

# TODO: compute test RMSE 
lin_rmse = np.sqrt(mean_squared_error(y_test, lin_pred))

# Check
assert isinstance(lin_model, Pipeline), "lin_model must be a sklearn Pipeline."
assert len(lin_pred) == len(y_test), "Number of predictions must match y_test length."
assert np.isfinite(lin_rmse) and lin_rmse > 0, "lin_rmse should be a positive finite number."
print("Step 7 check passed.")
print("Linear Regression Test RMSE:", lin_rmse)

Step 7 check passed.
Linear Regression Test RMSE: 2279047.5017167483


## 8) Cross-validation (no leakage)
Evaluate your **pipeline** using 5-fold cross-validation RMSE.

Reminder: because `lin_model` includes preprocessing + model in one Pipeline, `cross_val_score` will fit preprocessing only on each training fold (no leakage).

In [38]:
# TODO: cross-validation RMSE (no leakage)
# Hint: use cross_val_score(..., scoring="neg_mean_squared_error", cv=5)
lin_scores = cross_val_score(
    lin_model, 
    X_train, 
    y_train, 
    scoring="neg_mean_squared_error", 
    cv=5
)
lin_rmse_scores = np.sqrt(-lin_scores)

# Check
assert len(lin_rmse_scores) == 5, "You should have 5 CV RMSE scores for cv=5."
assert np.all(np.isfinite(lin_rmse_scores)), "All CV RMSE scores must be finite."
print("Step 8 check passed.")
print("Linear CV RMSE:", lin_rmse_scores)

Step 8 check passed.
Linear CV RMSE: [467477.80570854 484533.15918392 449612.59057272 522030.47140128
 427480.82437029]


## 9) Try two stronger models
Create two more pipelines:
- `tree_model` = DecisionTreeRegressor
- `forest_model` = RandomForestRegressor

### Requirements
- Both must include the same `preprocess` step.
- Set `random_state=42` for reproducibility.
- Evaluate both using 5-fold CV RMSE (same method as before).

In [39]:
# TODO: Decision Tree pipeline + CV RMSE
# Hint: DecisionTreeRegressor(random_state=42)

tree_model = Pipeline(steps=[
    ('preprocessor', preprocess),
    ('regressor', DecisionTreeRegressor(random_state=42))
])

tree_scores = cross_val_score(
    tree_model, 
    X_train, 
    y_train, 
    scoring="neg_mean_squared_error", 
    cv=5
)

tree_rmse_scores = np.sqrt(-tree_scores)

# Check
assert len(tree_rmse_scores) == 5, "You should have 5 CV RMSE scores for cv=5."
assert np.all(np.isfinite(tree_rmse_scores)), "All tree CV RMSE scores must be finite."
print("Step 9a check passed.")
print("Tree CV RMSE:", tree_rmse_scores)

Step 9a check passed.
Tree CV RMSE: [409813.40434195 373782.61869176 396790.13434936 361605.02567972
 371859.46436024]


In [40]:
# TODO: Random Forest pipeline + CV RMSE
# 1. Create the Random Forest pipeline
# We use the same 'preprocess' transformer and set a random_state for reproducibility
forest_model = Pipeline(steps=[
    ('preprocessor', preprocess),
    ('regressor', RandomForestRegressor(random_state=42))
])

# 2. Compute cross-validation RMSE
# Random Forests are computationally heavier, so this might take a few seconds longer
forest_scores = cross_val_score(
    forest_model, 
    X_train, 
    y_train, 
    scoring="neg_mean_squared_error", 
    cv=5
)

# 3. Convert to positive RMSE
forest_rmse_scores = np.sqrt(-forest_scores)

# Check
assert len(forest_rmse_scores) == 5, "You should have 5 CV RMSE scores for cv=5."
assert np.all(np.isfinite(forest_rmse_scores)), "All forest CV RMSE scores must be finite."
print("Step 9b check passed.")
print("Forest CV RMSE:", forest_rmse_scores)

Step 9b check passed.
Forest CV RMSE: [340579.88504531 263962.39978097 308688.6959962  273387.99735594
 287756.52059427]


## 10) Hyperparameter tuning with GridSearchCV
Tune the Random Forest **pipeline** using GridSearchCV.

### Task
1. Import `GridSearchCV`
2. Use a `param_grid` with parameters prefixed by `model__`
   - Example: `model__n_estimators`, `model__max_features`
3. Use `cv=3` to keep runtime reasonable
4. Fit on `X_train`, `y_train`
5. Print the best params and best RMSE (convert from negative MSE)

Tip: start with a small grid first (faster), then expand if needed.

In [41]:
# TODO: GridSearchCV over the Random Forest pipeline
# Hint: parameters should be prefixed with model__ (e.g., model__n_estimators)

from sklearn.model_selection import GridSearchCV

from sklearn.model_selection import GridSearchCV
# 1. Re-defining the pipeline so the step name matches the 'model__' prefix
forest_pipeline = Pipeline(steps=[
    ('preprocessor', preprocess),
    ('model', RandomForestRegressor(random_state=42))
])

param_grid = [
    {
        'model__n_estimators': [10, 30, 100], 
        'model__max_features': [2, 4, 6, 8]
    },
    {
        'model__bootstrap': [False], 
        'model__n_estimators': [3, 10], 
        'model__max_features': [2, 3, 4]
    },
]

grid_search = GridSearchCV(
    forest_pipeline, 
    param_grid, 
    cv=3, # Keep runtime reasonable
    scoring='neg_mean_squared_error',
    return_train_score=True
)

grid_search.fit(X_train, y_train)


# Check
if isinstance(param_grid, dict):
    grid_keys = list(param_grid.keys())
else:
    grid_keys = [k for d in param_grid for k in d.keys()]
assert any(k.startswith("model__") for k in grid_keys), "Grid keys must include the 'model__' prefix."
assert hasattr(grid_search, "best_estimator_"), "GridSearchCV did not fit correctly (no best_estimator_ found)."
best_rmse = np.sqrt(-grid_search.best_score_)
assert np.isfinite(best_rmse) and best_rmse > 0, "Best CV RMSE should be a positive finite number."
print("Step 10 check passed.")
print("Best params:", grid_search.best_params_)
print("Best CV RMSE:", best_rmse)

Step 10 check passed.
Best params: {'model__max_features': 6, 'model__n_estimators': 100}
Best CV RMSE: 352340.16199836816


## 11) Final evaluation on the test set
Choose your final model:
- If you did grid search: use `grid_search.best_estimator_`
- Otherwise: use `forest_model` (or your best model)

Then compute RMSE on `X_test` / `y_test`.

In [42]:
# TODO: pick final model

final_model = grid_search.best_estimator_

final_model.fit(X_train, y_train)

test_pred = final_model.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

# Check
assert len(test_pred) == len(y_test), "Number of test predictions must match y_test length."
assert np.isfinite(test_rmse) and test_rmse > 0, "test_rmse should be a positive finite number."
print("Step 11 check passed.")
print("Final Test RMSE:", test_rmse)

Step 11 check passed.
Final Test RMSE: 322444.6947423679
